<div style="background-color: #ADD8E6; border: 1px solid gray; padding: 3px">
    <h3>GraphRAG Index Generation</h3>
    The following is an overview of the workflow:
    <ul>
    <li>Uses Microsoft GraphRAG library to construct a GraphRAG index for the synthetically generated dataset:</li>
        <ul>
            <li>Uses reformatted code-to-summary pairs as input</li>
            <li>Uses openai/gpt-oss-20b at the chat model</li>
            <li>Uses intfloat/e5-mistral-7b-instruct as the embedding model</li>
        </ul>
    </li>
    <li>Stores index in LanceDB database backed by Minio bucket</li>
    </ul>
</div>

In [1]:
##############################################
# Imports
##############################################
from minio import Minio
import os
import lancedb
import traceback
from minio.error import S3Error
import subprocess
import tracemalloc
tracemalloc.start()
import nest_asyncio
nest_asyncio.apply()
import utils

In [9]:
##############################################
# Generate GraphRAG index and store in LanceDB
##############################################

def generate_graphrag_index(dataset_name: str,
                            graphrag_source_path: str,
                            jsonl_source_path: str) :

    """
    Splits the provided jsonl file into seprate json files, then
    generates a GraphRAG index from the json files.
    Args:
        dataset_name: The source dataset name
        graphrag_source_path: The source path used by the GraphRAG index configuration
        jsonl_source_path: The jsonl source file
    Returns:
        None
    """

    ##############################################
    # Imports
    ##############################################
    from minio import Minio
    import os
    import lancedb
    from datasets import load_dataset
    import traceback
    import subprocess
    import tracemalloc
    tracemalloc.start()
    import nest_asyncio
    nest_asyncio.apply()
    import utils

    try:

        graph_rag_config_path = f"{graphrag_source_path}/settings.yaml"
        
        os.makedirs(f"{graphrag_source_path}/input", exist_ok=True)

        os.makedirs(f"{graphrag_source_path}/output", exist_ok=True)
        
        os.makedirs(os.path.dirname(jsonl_source_path), exist_ok=True)
        
        print(f"Preprocessing dataset {dataset_name}...")
        
        updated_dataset = utils.postprocess_dataset(dataset_name,
                                                    jsonl_source_path)

        print(f"Splitting dataset {dataset_name} into json files...")
    
        utils.split_jsonl_into_json_files(jsonl_source_path, f"{graphrag_source_path}/input")

        print("Running index...")
    
        result = subprocess.run(["bash", "graphrag.sh", graphrag_source_path, graph_rag_config_path], capture_output=True, text=True, check=False)
            
        print(f"\nSubprocess output: {result.stdout}")
        
        if result.stderr:
            
            raise Exception(f"Error processing GraphRAG command: {result.stderr}")
        
    except Exception as e:
        
        print(f"Error processing GraphRAG DB: {e}")
        traceback.print_exc()

In [3]:
##############################################
# Upload data to LanceDB
##############################################
def upload_graphrag_index_to_lancedb(graphrag_source_path: str, lancedb_minio_bucket_name: str, lancedb_db_name: str):
    """
    Uploads the GraphRAG index from the provided source path to the specified minio bucket.
    (Requires a valid Minio configuration which has been preconfigured using environment variables.)
    Args:
        graphrag_source_path: The source path for the GraphRAG index files
        lancedb_minio_bucket_name: The backing Minio bucket for the LanceDB database.
        lancedb_db_name: The LanceDB database name.
    Returns:
        None
    """

    ##############################################
    # Imports
    ##############################################
    from minio import Minio
    import os
    import lancedb
    from datasets import load_dataset
    import nest_asyncio
    import pandas as pd
    nest_asyncio.apply()
    
    
    try:

        print(f"Uploading index from {graphrag_source_path}...")
        
        graphrag_index_source_path = f"{graphrag_source_path}"
        
        db = lancedb.connect(f"s3://{lancedb_minio_bucket_name}/{lancedb_db_name}",
                             
            storage_options={
                "endpoint_url": os.getenv("AWS_S3_ENDPOINT"),
                
                "aws_access_key_id": os.getenv("AWS_ACCESS_KEY_ID"),
                
                "aws_secret_access_key": os.getenv("AWS_SECRET_ACCESS_KEY"),
                
                "s3_force_path_style": "true",
                
                "allow_http": "true",
            }
        )
    
        local_db = lancedb.connect(f"{graphrag_source_path}/lancedb")
    
        all_tables = local_db.table_names()
    
        # Migrate Global Search tables
        print("Migrating global search tables...")
        
        for table_name in all_tables:
    
            try:
    
                local_table = local_db.open_table(table_name)
        
                local_df = local_table.to_pandas()
        
                db.create_table(table_name, data=local_df)
        
                print(f"{local_table} migrated.")
        
            except Exception as e:
                
                print(f"Error processing GraphRAG migration to Minio: {e}") 
    
        # Migrate Local Search tables
        print("Migrating local search tables...")
        
        for file_path in os.listdir(f"{graphrag_index_source_path}"):
            
            if file_path.endswith(".parquet"):
    
                try:
            
                    full_path = os.path.join(graphrag_index_source_path, file_path)
        
                    local_df = pd.read_parquet(full_path)
    
                    table_name = file_path.split(".", 1)[0]
        
                    db.create_table(table_name, data=local_df)
        
                    print(f"{table_name} migrated.")
        
                except Exception as e:
                    
                    print(f"Error processing GraphRAG migration to Minio: {e}")    
            
        print("Migration complete.")
        
    except Exception as e:
        
        print(f"Error processing GraphRAG migration to Minio: {e}")

### Generate GraphDB Index and Store in LanceDB
Start the indexing pipeline!

In [4]:
##############################################
# ColdFusion Baseline Index
##############################################
# try:
#     graphrag_path = "baseline_graph_rag/source"
    
#     jsonl_source_path = "baseline_graph_rag/json/graphrag.jsonl"
    
#     baseline_dataset_name = "oaawofolu/emerson"
    
#     bucket_name = "data"
    
#     baseline_lancedb_db_name = "cfcode-baseline"
    
#     generate_graphrag_index(baseline_dataset_name,
#                             graphrag_path,
#                             jsonl_source_path)
    
#     upload_graphrag_index_to_lancedb(f"{graphrag_path}/output", 
#                                      bucket_name, 
#                                      baseline_lancedb_db_name)
# except Exception as e:
        
#         print(f"Error processing ColdFusion Baselinex Index: {e}")

#         traceback.print_exc()

Preprocessing dataset oaawofolu/emerson...


Map:   0%|          | 0/1796 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Splitting dataset oaawofolu/emerson into json files...
Successfully converted 'baseline_graph_rag/json/graphrag.jsonl' to json files under'baseline_graph_rag/source/input'.
Running index...

Subprocess output: Initializing GraphRAG index...
Copying settings.yaml...
Configuring prompts...
/opt/app-root/lib64/python3.11/site-packages/litellm/llms/custom_httpx/async_client_cleanup.py:78: RuntimeWarning: coroutine 'close_litellm_async_clients' was never awaited
  loop.close()
Populating GraphRAG index...
Starting pipeline with workflows: load_input_documents, create_base_text_units, create_final_documents, extract_graph, finalize_graph, extract_covariates, create_communities, create_final_text_units, create_community_reports, generate_text_embeddings
Starting workflow: load_input_documents

Workflow complete: load_input_documents
Starting workflow: create_base_text_units
  1 / 1 ............................................................................................
  1 / 1 ...........

In [ ]:
##############################################
# ColdFusion Sample Codebase Index
##############################################
try:
    graphrag_path = "graph_rag/source"
    
    jsonl_source_path = "graph_rag/json/graphrag.jsonl"
    
    prompt_dataset_name = "oaawofolu/cfcode-golfap"
    
    bucket_name = "data"
    
    prompt_lancedb_db_name = "cfcode-golfap-idx"
    
    generate_graphrag_index(prompt_dataset_name,
                            graphrag_path,
                            jsonl_source_path)
    
    upload_graphrag_index_to_lancedb(f"{graphrag_path}/output", 
                                     bucket_name, 
                                     prompt_lancedb_db_name)

except Exception as e:
        
        print(f"Error processing ColdFusion Sample Codebase Index: {e}")

        traceback.print_exc()

### GraphRAG Queries
Perform queries against the GraphRAG index

In [2]:
input1 = """
This code represents an application called Golfap, which is 
a social golf scorekeeping app that makes it easy to track scores, bets, and games with friends.
"""

prompt1 = """
Your task is to analyze this code and generate a software design document.
This document should include a concise explanation of the purpose of the code.
If the provided snippet does not appear to be valid code, indicate that this is not valid code.
Stop after you have finished writing the 3 sections described below.
Do not stray from the functionality provided in the codebase. Stick strictly to the code provided.


Your analysis must include the following sections:

**1. Summary:** 
* Provide a clear and concise explanation of the purpose of the code.

**2. Components:** 
* Provide a concise list of all the important ColdFusion relevant components you can find. 
* Do not rename any components or change their case; provide them exactly as they are shown in the code.
* Include the following:
    * List of primary modules and their responsibilities: include the file path to the modules if it is available
    * List of primary web pages and their responsibilitites: include the file path to the web pages if it is available
    * List of ColdFusion components and their responsibilities: include the file path to the components if it is available
    * List of reusable templates and their responsibilities: include the file path to the templates if it is available
    * List of the names of custom components and their responsibilities (if any), exactly as shown in the code: cfcomponents, user-defined functions and any other custom components you can find; include the file path to the templates if it is available
    
**3. Domain:** 
* Generate a concise outline of the domain model associated with this code.
* Include the current state of the domain objects based on information extracted from the code.

"""

prompt2 = """
Here is a summary of the codebase:
Generate a language-agnostic Software Design Document for this code. 
The document should be structured, professional, and suitable for both technical and non-technical stakeholders.
Also exclude any details that link the requirements to ColdFusion or any other specific programming language or framework.
Instead, it should focus on universal concepts and architecture that can be implemented in any programming language.
Stop after you have finished writing the 3 sections described below.

The SDD must include the following sections:

**1. System Architecture**
*   **Overall Design:** Describe the main architectural patterns used and how the different components interact. 
*   **Key Components:** List of the primary modules, classes, domain model and services and their responsibilities.

**2. Functional Requirements**
*   **Input Handling:** How does the system accept inputs?
*   **Data Processing:** List of the main logic, algorithms, and data transformations.
*   **Output Generation:** How are results produced and presented?

**3. Business Requirements**
*   **Business Rules:** List of specific rules that govern how the business operates, which the software must enforce.
*   **Success Criteria:** List of measurable criteria to determine if the project is successful, also known as acceptance criteria.

Only use the context provided in the summary above. Do not stray from the context provided in the summary. 
Include named components and objects based on the context where it makes sense.
"""

### FULL PIPELINE
Execute full pipeline! 

In [4]:
##############################################
# Full Pipeline
##############################################
def graphrag_pipeline(git_repo: str, app_name: str):
    output = None
    
    try:
        
        graphrag_path = f"graph_rag_{app_name}/source"
        
        jsonl_source_path = f"graph_rag_{app_name}/json/graphrag.jsonl"
        
        prompt_dataset_name = f"oaawofolu/cfcode-{app_name}"
        
        bucket_name = "data"
        
        prompt_lancedb_db_name = f"cfcode-{app_name}-idx"
    
        local_lancedb_path = f"local-lancedb-{app_name}"
    
        utils.create_or_update_indexing_job(git_repo, bucket_name)
        
        generate_graphrag_index(prompt_dataset_name,
                                graphrag_path,
                                jsonl_source_path)
        
        upload_graphrag_index_to_lancedb(f"{graphrag_path}/output", 
                                         bucket_name, 
                                         prompt_lancedb_db_name)
    
        utils.download_lancedb_index(bucket_name, prompt_lancedb_db_name, local_lancedb_path)
        
        output = utils.query_lancedb_graphrag_index(root=local_lancedb_path, 
                                           config=f"settings.yaml", 
                                           prompt=f"{input1}\n{prompt1}", 
                                           response_type="text format",
                                           method="global")
    
        utils.create_or_update_indexing_job(git_repo, bucket_name, results=output)
    
        
    
    except Exception as e:
            
            print(f"Error processing ColdFusion Sample Codebase Index: {e}")
    
            traceback.print_exc()


Copying table: communities
Copying table: community_reports
Copying table: default-community-full_content
Copying table: default-entity-description
Copying table: default-text_unit-text
Copying table: documents
Copying table: entities
Copying table: relationships
Copying table: text_units
DB index initialization complete...

Subprocess output========================================= **1. Summary**  
Golfap is a ColdFusion‑based social golf score‑keeping application.  It lets users record round scores, place bets on tournament outcomes, and manage games with friends.  The code also aggregates live golf news from multiple RSS/Atom feeds, displays tournament leaderboards, and provides a dynamic web interface that updates scores in real time.  The application relies on a GOLFAP datasource, a set of CFQUERY tags, and several ColdFusion components for JSON serialization and business logic.  The overall purpose is to give golfers a single web portal for score tracking, betting, and news consu

In [ ]:
graphrag_pipeline("https://github.com/holtonma/cf_golfap.git", "golfap")
graphrag_pipeline("https://github.com/kishore31/CheckMate-CMS", "checkmatecms")
graphrag_pipeline("https://github.com/ehynds/cfml", "ehynds")
graphrag_pipeline("https://github.com/timblair/cflipsum", "cflipsum")
graphrag_pipeline("https://github.com/illuminerdi/fusebox_implicit_skelly", "fusebox")

In [2]:
utils.fetch_indexing_job("https://github.com/holtonma/cf_golfap.git", "data")

'**1. Summary**  \nGolfap is a ColdFusion‑based social golf score‑keeping application.  It lets users record round scores, place bets on tournament outcomes, and manage games with friends.  The code also aggregates live golf news from multiple RSS/Atom feeds, displays tournament leaderboards, and provides a dynamic web interface that updates scores in real time.  The application relies on a GOLFAP datasource, a set of CFQUERY tags, and several ColdFusion components for JSON serialization and business logic.  The overall purpose is to give golfers a single web portal for score tracking, betting, and news consumption.  \n\n**2. Components**  \n\n| Category | Component | File Path (if available) | Responsibility | Reference |\n|----------|-----------|--------------------------|----------------|-----------|\n| **Primary modules** | `GOLFAP.com` | `/GOLFAP.com` | Main entry point; loads header, footer, news widgets, and navigation. | [Data: Reports (201, 205, 27, 110, 246, +more)] |\n| | `G